# [EDA] 2022년도 전체 시도 세부사업 예산현황 분리·정제 (작업중) — 이슈 #8

> - 입력: `data/raw/칼럼정렬/*2022*.xlsx`의 `정리본_자동`·`Table 1` 시트.
> - QA는 원시값 완전 동일 비교와 소수점 첫째 자리 반올림 후 차이 0.0 비교를 구분한다.
> - 서울 원본행 28의 연속 후보는 반복 머리글을 건너뛴 앞 실제 데이터행인 원본행 24에 병합하고 leaf에서 제외한다.
> - 후보 판정과 구조 이상 7행 제외 후 최종 leaf는 8,122행이며, long 변환 시 16,244행이다.
> - LLM 정제를 실행했으며, 319건 전수검토 결과를 체크포인트와 최종 산출물에 반영했다.


In [1]:
# 라이브러리 임포트
import re
from pathlib import Path

import pandas as pd
import numpy as np

YEAR = 2022  # 시행계획 연도
PREVIOUS_YEAR = YEAR - 1
CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"

# 경로 설정
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORTS_DIR = Path("../reports")


def get_sido_dir(sido: str) -> Path:
    """시도별 중간 산출물 디렉터리를 생성하고 경로를 반환한다."""
    sido_dir = INTERIM_DIR / sido
    sido_dir.mkdir(parents=True, exist_ok=True)
    return sido_dir


def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마·공백을 정리하고 '-', '비예산'을 0으로 치환한 뒤 숫자로 변환한다."""
    normalized = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"-": "0", "비예산": "0"})
    )
    return pd.to_numeric(normalized, errors="coerce")


# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

In [2]:
# 칼럼정렬 파일에는 파이프라인 표준 컬럼과 시트가 준비되어 있음.
COLUMN_ALIGNED_DIR = DATA_DIR / "칼럼정렬"
source_file = next(COLUMN_ALIGNED_DIR.glob(f"*{YEAR}*.xlsx"))
df_raw = pd.read_excel(source_file, sheet_name="정리본_자동")

print(df_raw.shape)
df_raw.head(10)

(8472, 12)


,지역,세부사업명,사업분류재정구분,2022년 예산,2021년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,7299445,6946037,353408,5.1,NaN,1,2,4,데이터행
1,서울,1. 함께 일하고 함께 돌보는 사회(공통),NaN,2991826,3085406,-93580,-3,NaN,1,2,5,데이터행
2,서울,저출생 인식개선 캠페인,공통,58,43,15,34.9,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",1,2,6,데이터행
3,서울,어린이 안전 영상정보 인프라 구축,공통,38039,33280,4759,14.3,지원대상 : 어린이\n지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,1,2,7,데이터행
4,서울,입양아동 가족지원,공통,4780,4120,660,16,"지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정\n지원내용 : 입양아동양육보조금, 심리치료비 등",1,2,8,데이터행
5,서울,국공립어린이집 등 보육서비스 수준제고,공통,330577,322975,7602,2.4,지원대상 : 국공립어린이집 등\n지원내용 : 보육교직원 인건비 지원,1,2,9,데이터행
6,서울,국공립어린이집 확충,공통,15523,40000,-24477,-61.2,국공립어린이집 이용률 46%달성,1,2,10,데이터행
7,서울,어린이집 이용 영유아 무상보육 확대,공통,791160,854364,-63204,-7.4,지원대상 : 어린이집 이용 영유아\n지원내용 : 해당 연령 보육료,1,2,11,데이터행
8,서울,육아종합지원센터 운영,공통,6293,6104,189,3.1,"지원대상 : 보육교직원 및 영유아 부모\n지원내용 : 보육정보 및 교육, 전문상담 제공",1,2,12,데이터행
9,서울,가정양육수당 지원,공통,115422,171371,-55949,-32.6,지원대상 : 어린이집･유치원 미이용 0~86개월 미만 아동\n지원내용 :10~20만원(월) 양육수당 지원,1,2,13,데이터행


In [3]:
# 기본검사
print("[데이터셋 크기]\n", df_raw.shape)
print("=====================================")
print("[데이터 타입]\n", df_raw.dtypes)
print("=====================================")
print("[각 컬럼별 결측치 개수 평균]\n", df_raw.isna().mean().round(3))
print("=====================================")
print("[중복 행 수]", df_raw.duplicated().sum())
print("=====================================")

# 지역(시도) 목록 확인
print("[지역(시도) 목록 확인]\n", df_raw["지역"].value_counts())

[데이터셋 크기]
 (8472, 12)
[데이터 타입]
 지역             str
세부사업명          str
사업분류재정구분       str
2022년 예산    object
2021년 예산    object
증감액         object
비율          object
주요내용           str
원본표구간        int64
머리글행         int64
원본행          int64
행유형            str
dtype: object
[각 컬럼별 결측치 개수 평균]
 지역          0.000
세부사업명       0.000
사업분류재정구분    0.020
2022년 예산    0.003
2021년 예산    0.005
증감액         0.007
비율          0.000
주요내용        0.021
원본표구간       0.000
머리글행        0.000
원본행         0.000
행유형         0.000
dtype: float64
[중복 행 수] 0
[지역(시도) 목록 확인]
 지역
경기    1348
부산     773
경북     768
경남     733
충남     645
전남     573
강원     542
전북     519
충북     514
인천     398
대전     339
대구     294
울산     270
서울     244
광주     217
제주     174
세종     121
Name: count, dtype: int64


In [4]:
# 지역별로 데이터 분리
sido_dfs = {sido: group.copy() for sido, group in df_raw.groupby("지역")}

# 시도별 행 수 확인
for sido, df_sido in sido_dfs.items():
    print(sido, len(df_sido))

강원 542
경기 1348
경남 733
경북 768
광주 217
대구 294
대전 339
부산 773
서울 244
세종 121
울산 270
인천 398
전남 573
전북 519
제주 174
충남 645
충북 514


In [5]:
# 이전 데이터셋에서 확인한 classify_row 적용
sido_title_pattern = re.compile(r"붙\s*임\s*\(([^)]+)\)")
# 2022 페이지 반복헤더/시도경계 제목행
header_noise_pattern = re.compile(
    r"고령사회기본계획|지방자치단체\s*시행계획|세부사업별\s*예산\s*현황"
)


def classify_row(세부사업명: str) -> str:
    """대분류_소계 / 중분류_소계 / 헤더반복 / 세부사업 행 판별"""
    if pd.isna(세부사업명) or str(세부사업명).strip() == "":
        return "헤더반복"

    name = str(세부사업명).strip()

    if name == "세부사업명":
        return "헤더반복"
    if sido_title_pattern.search(name):
        return "헤더반복"
    if header_noise_pattern.search(name):
        return "헤더반복"
    if re.match(
        r"^[Ⅰ-Ⅿ]", name
    ):  # 유니코드 로마숫자 대문자 전체 블록(Ⅰ~ⅬⅭⅮⅯ) - 대분류 10개(Ⅹ) 초과 대비
        return "대분류_소계"
    if re.match(r"^\d+\.", name) and re.search(r"\((공통|자체)\)$", name):  # 조건 추가
        return "중분류_소계"
    return "세부사업"


df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)
df_raw[CURRENT_BUDGET_COL] = to_numeric_budget(df_raw[CURRENT_BUDGET_COL])

In [6]:
df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 시도별 행종류 분포
rowtype_distribution = df_raw.groupby(["지역", "사업행구분"]).size().unstack(fill_value=0)
rowtype_distribution

사업행구분,대분류_소계,세부사업,중분류_소계,헤더반복
지역,,,,
강원,2,521,8,11
경기,2,1311,8,27
경남,2,707,8,16
경북,2,742,8,16
광주,2,204,7,4
대구,2,278,8,6
대전,2,322,8,7
부산,2,748,8,15
서울,2,228,8,6


-> 2022년 문서에서는 모든 시도에 페이지 반복 머리글·시도 경계 제목행이 있어 `헤더반복`이 지역별 3~27건 탐지되었다. 경기가 27건으로 가장 많고, 세종이 3건으로 가장 적으며, 서울은 6건이다.

-> 헤더 반복 및 경계 제목행은 세부사업 분석 대상에서 제외한다.


In [7]:
# 광주 중분류_소계 행 확인
df_gwangju = sido_dfs["광주"]
df_gwangju["사업행구분"] = df_gwangju["세부사업명"].apply(classify_row)
print("[광주] 중분류_소계 행 확인")
print("==================================================")
print(df_gwangju[df_gwangju["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

print("\n")

# 전남도 동일하게 확인
df_jeongnam = sido_dfs["전남"]
df_jeongnam["사업행구분"] = df_jeongnam["세부사업명"].apply(classify_row)
print("[전남] 중분류_소계 행 확인")
print("==================================================")
print(df_jeongnam[df_jeongnam["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

[광주] 중분류_소계 행 확인
1711       1. 함께 일하고 함께 돌보는 사회(공통)
1723         2. 건강하고 능동적인 고령사회(공통)
1735     3. 모두의 역량이 고루 발휘되는 사회(공통)
1746      1. 함께 일하고 함께 돌보는\n사회(자체)
1838      2. 건강하고 능동적인 고령사회 구축(자체)
1877    3. 모두의 역량이 고루\n발휘되는 사회(자체)
1914          4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


[전남] 중분류_소계 행 확인
6226       1. 함께 일하고 함께 돌보는 사회(공통)
6235         2. 건강하고 능동적인 고령사회(공통)
6255         4.인구구조 변화에 대한\n적응(공통)
6270       1. 함께 일하고 함께 돌보는 사회(자체)
6528     2. 건강하고 능동적인\n고령사회 구축(자체)
6645    3. 모두의 역량이 고루\n발휘되는 사회(자체)
6740         4.인구구조 변화에 대한\n적응(자체)
Name: 세부사업명, dtype: str


-> 광주는 `4. 인구구조 변화에 대한 적응(공통)`이 없고,

-> 전남은 `3. 모두의 역량이 고루 발휘되는 사회(공통)`가 없다.

-> 출력된 7개 항목은 모두 정상적인 중분류 형식이므로, 누락된 영역이 원문에도 없는지 `Table 1` 시트와 대조해 원문 구성 차이인지 분류 누락인지를 확인할 필요가 있다.


In [8]:
# Table 1 시트의 시도별 원문 구간에서 누락된 중분류를 대조한다.
df_table1 = pd.read_excel(source_file, sheet_name="Table 1", header=None, dtype="string")
table1_row_text = df_table1.fillna("").agg(" ".join, axis=1).str.replace(r"\s+", "", regex=True)

# `(OO시도·OO교육청)` 형식의 행을 시도 구간 시작점으로 사용한다.
region_header_pattern = re.compile(r"\(([^()･]+)･[^()]*교육청\)")
region_starts = []
for row_idx, row_text in table1_row_text.items():
    match = region_header_pattern.search(row_text)
    if match:
        region_starts.append((row_idx, match.group(1)))

table1_qa_targets = {
    "광주광역시": "4. 인구구조 변화에 대한 적응(공통)",
    "전라남도": "3. 모두의 역량이 고루 발휘되는 사회(공통)",
}

for region_name, target_title in table1_qa_targets.items():
    start_pos = next(i for i, (_, name) in enumerate(region_starts) if name == region_name)
    start_row = region_starts[start_pos][0]
    end_row = (
        region_starts[start_pos + 1][0] if start_pos + 1 < len(region_starts) else len(df_table1)
    )
    region_text = table1_row_text.iloc[start_row:end_row]
    normalized_title = re.sub(r"\s+", "", target_title)
    exact_rows = region_text[region_text.str.contains(re.escape(normalized_title), regex=True)]

    # 번호·공통 표기가 다른 사례도 잡기 위해 핵심 명칭으로 한 번 더 검색한다.
    core_title = re.sub(r"^\d+\.", "", normalized_title).replace("(공통)", "")
    candidate_rows = region_text[region_text.str.contains(re.escape(core_title), regex=True)]

    print(f"[{region_name}] Table 1 행 구간: {start_row + 1}~{end_row}")
    print(f"정확 명칭 일치: {len(exact_rows)}건 / 핵심 명칭 후보: {len(candidate_rows)}건")
    if candidate_rows.empty:
        print(f"원문 구간에서 '{target_title}'을(를) 찾지 못함")
    else:
        display(df_table1.loc[candidate_rows.index].dropna(axis=1, how="all"))
    print("=" * 70)


# 2022 행유형 연속 후보 1건: 지역·원본행 키 기반 판정과 원본 문맥 진단
rowtype_decisions = pd.DataFrame(
    [
        {
            "지역": "서울",
            "후보_원본행": 28,
            "판정": "앞 행 병합",
            "병합대상_원본행": 24,
            "병합후_세부사업명": "학대피해아동 보호를 위한 안전망 구축",
            "판정근거": (
                "원본행 25~27은 페이지 제목·반복 머리글이고, 원본행 28은 "
                "앞 실제 데이터행 원본행 24의 세부사업명과 주요내용이 이어지는 행이다."
            ),
        }
    ]
)
decision_key_cols = ["지역", "후보_원본행"]
assert not rowtype_decisions.duplicated(decision_key_cols).any()

decision = rowtype_decisions.iloc[0]
candidate_mask = df_raw["지역"].eq(decision["지역"]) & df_raw["원본행"].eq(decision["후보_원본행"])
rowtype_candidates = df_raw.loc[candidate_mask].copy()
assert len(rowtype_candidates) == 1, (
    f"2022 행유형 연속 후보는 정확히 1건이어야 합니다: {len(rowtype_candidates)}건"
)
assert rowtype_candidates.iloc[0]["행유형"] == "구분행사업명 연속 후보"

candidate_idx = rowtype_candidates.index[0]
candidate_pos = df_raw.index.get_loc(candidate_idx)
region_rows = df_raw[df_raw["지역"].eq(decision["지역"])].sort_values("원본행")
actual_data_mask = region_rows["행유형"].eq("데이터행") & region_rows["사업행구분"].eq("세부사업")
previous_leaf = region_rows[
    actual_data_mask & region_rows["원본행"].lt(decision["후보_원본행"])
].tail(1)
next_leaf = region_rows[actual_data_mask & region_rows["원본행"].gt(decision["후보_원본행"])].head(
    1
)
assert len(previous_leaf) == 1 and len(next_leaf) == 1
assert int(previous_leaf.iloc[0]["원본행"]) == int(decision["병합대상_원본행"])

target_mask = df_raw["지역"].eq(decision["지역"]) & df_raw["원본행"].eq(decision["병합대상_원본행"])
merge_target = df_raw.loc[target_mask].copy()
assert len(merge_target) == 1, (
    f"병합 대상 원본행 24는 정확히 1건이어야 합니다: {len(merge_target)}건"
)

previous_pos = df_raw.index.get_loc(previous_leaf.index[0])
boundary_rows = df_raw.iloc[previous_pos + 1 : candidate_pos]

candidate_context = pd.concat(
    [
        previous_leaf.assign(진단위치="앞 세부사업행"),
        boundary_rows.assign(진단위치="페이지 경계행"),
        rowtype_candidates.assign(진단위치="연속 후보행"),
        next_leaf.assign(진단위치="뒤 세부사업행"),
    ]
)
context_cols = [
    "진단위치",
    "지역",
    "원본행",
    "행유형",
    "사업행구분",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
]
display(candidate_context[context_cols])

table1_start_row = int(previous_leaf.iloc[0]["원본행"])
table1_end_row = int(next_leaf.iloc[0]["원본행"])
table1_candidate_context = df_table1.iloc[table1_start_row - 1 : table1_end_row].copy()
table1_candidate_context.insert(
    0,
    "Table1_엑셀행",
    range(table1_start_row, table1_end_row + 1),
)
table1_candidate_context = table1_candidate_context.dropna(axis=1, how="all")
display(table1_candidate_context)

# 원본행 24의 예산·재정구분 값은 보존하고, 이름과 실제 주요내용만 병합한다.
preserved_target_cols = [
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "증감액",
    "비율",
    "사업분류재정구분",
]
target_values_before = merge_target.iloc[0][preserved_target_cols].copy()
target_name_before = str(merge_target.iloc[0]["세부사업명"])
target_content_before = merge_target.iloc[0]["주요내용"]
candidate_name = str(rowtype_candidates.iloc[0]["세부사업명"])
candidate_content = rowtype_candidates.iloc[0]["주요내용"]


def has_actual_text(value) -> bool:
    """결측·빈 문자열·대시가 아닌 실제 텍스트인지 판별한다."""
    return pd.notna(value) and str(value).strip() not in {"", "-"}


content_parts = []
if has_actual_text(target_content_before):
    content_parts.append(str(target_content_before).strip())
if has_actual_text(candidate_content):
    content_parts.append(str(candidate_content).strip())
merged_content = "\n".join(content_parts) if content_parts else pd.NA

df_raw.loc[target_mask, "세부사업명"] = decision["병합후_세부사업명"]
df_raw.loc[target_mask, "주요내용"] = merged_content
df_raw["후보판정_제외"] = False
df_raw.loc[candidate_mask, "후보판정_제외"] = True

merged_target = df_raw.loc[target_mask]
assert len(merged_target) == 1
assert merged_target.iloc[0]["세부사업명"] == "학대피해아동 보호를 위한 안전망 구축"
pd.testing.assert_series_equal(
    merged_target.iloc[0][preserved_target_cols],
    target_values_before,
    check_names=False,
)
assert "위한 안전망 구축" not in str(candidate_content)
assert df_raw.loc[candidate_mask, "후보판정_제외"].sum() == 1

rowtype_review = pd.DataFrame(
    [
        {
            "지역": decision["지역"],
            "후보_원본행": int(decision["후보_원본행"]),
            "앞_데이터행_원본행": int(previous_leaf.iloc[0]["원본행"]),
            "뒤_데이터행_원본행": int(next_leaf.iloc[0]["원본행"]),
            "후보_세부사업명": candidate_name,
            "판정": decision["판정"],
            "병합대상_원본행": int(decision["병합대상_원본행"]),
            "병합전_세부사업명": target_name_before,
            "병합후_세부사업명": merged_target.iloc[0]["세부사업명"],
            "판정근거": decision["판정근거"],
        }
    ]
)
assert len(rowtype_review) == 1
assert rowtype_review.iloc[0]["판정"] == "앞 행 병합"
assert rowtype_review.iloc[0]["병합대상_원본행"] == 24
rowtype_review.to_csv(
    REPORTS_DIR / f"{YEAR}_행유형_후보_검토.csv",
    index=False,
    encoding="utf-8-sig",
)
display(rowtype_review)

[광주광역시] Table 1 행 구간: 1968~2210
정확 명칭 일치: 0건 / 핵심 명칭 후보: 1건


,1,10,14,19,24
2195,4.인구구조 변화에 대한 적응(자체),158,110,48,43.6


[전라남도] Table 1 행 구간: 7112~7761
정확 명칭 일치: 0건 / 핵심 명칭 후보: 1건


,1,11,15,20,26
7585,3. 모두의 역량이 고루\n발휘되는 사회(자체),317359.5,117935.6,199423.9,169.1


,진단위치,지역,원본행,행유형,사업행구분,세부사업명,사업분류재정구분,2022년 예산,2021년 예산,주요내용
20,앞 세부사업행,서울,24,데이터행,세부사업,학대피해아동 보호를,공통,9586.0,7831,"지원대상 : 아동보호전문기관(10개소),"
21,페이지 경계행,서울,25,데이터행,헤더반복,제4차 저출산･고령사회기본계획 2022년도 지방자치단체 시행계획(총괄),제4차 저출산･고령사회기본계획 2022년도 지방자치단체 시행계획(총괄),NaN,제4차 저출산･고령사회기본계획 2022년도 지방자치단체 시행계획(총괄),제4차 저출산･고령사회기본계획 2022년도 지방자치단체 시행계획(총괄)
22,연속 후보행,서울,28,구분행사업명 연속 후보,세부사업,위한 안전망 구축,NaN,NaN,NaN,"학대피해아동쉼터(6개소) 운영지원 지원내용 :\n학대받은 아동 발견, 보호, 치료 등 전문서비스 및 아동학대예방 교육 및 홍보\n피해아동 보호 및 숙식 제공, 생활지원, 상담과 치료 제공"
23,뒤 세부사업행,서울,29,데이터행,세부사업,인터넷중독예방상담센터 운영,공통,4123.0,4123,"지원대상 : 청소년 및 그 가족\n지원내용 : 인터넷 스마트폰 과의존 예방교육 및 상담,\n치유 프로그램 개발 보급 및 지도자 양성 등"


,Table1_엑셀행,0,2,4,5,8,9,12,13,16,18,21,23,26
23,24,학대피해아동 보호를,공통,9586,<NA>,7831,<NA>,1755,<NA>,22.4,<NA>,"지원대상 : 아동보호전문기관(10개소),",<NA>,<NA>
24,25,제4차 저출산･고령사회기본계획 2022년도 지방자치단체 시행계획(총괄),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
25,26,세부사업명,<NA>,<NA>,사업분류,<NA>,2022년\n예산\n(a),<NA>,2021년\n예산\n(b),<NA>,증감내역,<NA>,<NA>,주요내용
26,27,<NA>,<NA>,<NA>,공통\n/자체,<NA>,<NA>,<NA>,<NA>,<NA>,증감액\n(a-b),<NA>,비율,<NA>
27,28,위한 안전망 구축,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"학대피해아동쉼터(6개소) 운영지원 지원내용 :\n학대받은 아동 발견, 보호, 치료 등 전문서비스 및 아동학대예방 교육 및 홍보\n피해아동 보호 및 숙식 제공, 생활지원, 상담과 치료 제공"
28,29,인터넷중독예방상담센터 운영,<NA>,<NA>,공통,<NA>,4123,<NA>,4123,<NA>,-,<NA>,-,"지원대상 : 청소년 및 그 가족\n지원내용 : 인터넷 스마트폰 과의존 예방교육 및 상담,\n치유 프로그램 개발 보급 및 지도자 양성 등"


,지역,후보_원본행,앞_데이터행_원본행,뒤_데이터행_원본행,후보_세부사업명,판정,병합대상_원본행,병합전_세부사업명,병합후_세부사업명,판정근거
0,서울,28,24,29,위한 안전망 구축,앞 행 병합,24,학대피해아동 보호를,학대피해아동 보호를 위한 안전망 구축,"원본행 25~27은 페이지 제목·반복 머리글이고, 원본행 28은 앞 실제 데이터행 원본행 24의 세부사업명과 주요내용이 이어지는 행이다."


-> 실제 원본 파일에도 없는 것 확인

-> 서울 원본행 28은 반복 머리글 원본행 25~27을 건너뛴 앞 실제 데이터행 원본행 24에 병합했다. 후보 행과 구조 이상 7행을 제외한 최종 수치는 leaf 8,122행, long 16,244행이다.


In [9]:
# 지역별로 원본 순서 유지한 채 대분류/중분류 라벨 세부사업행에 전파
def assign_labels(df_sido: pd.DataFrame) -> pd.DataFrame:
    """대분류_소계 / 중분류_소계 행의 이름을 뒤따르는 세부사업 행에 전파"""
    df_sido = df_sido.sort_values(
        "원본행"
    ).copy()  # 그냥 copy해도 되지만(raw도 이미 순서가 정렬되어있음) ffill이 순서에 의존하므로 원본 문서 순서를 명시적으로 보장함.
    major_mask = df_sido["사업행구분"] == "대분류_소계"
    medium_mask = df_sido["사업행구분"] == "중분류_소계"
    df_sido["대분류"] = df_sido["세부사업명"].where(major_mask).ffill()
    df_sido["중분류"] = df_sido["세부사업명"].where(medium_mask).ffill()

    return df_sido


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# leaf 후보 중 계층 라벨이 없거나 공통/자체 구분이 아닌 행은 QA 목록으로 남긴다.
leaf_candidate_mask = df_labeled["사업행구분"].eq("세부사업") & ~df_labeled["후보판정_제외"]
structure_invalid_mask = (
    df_labeled["대분류"].isna()
    | df_labeled["중분류"].isna()
    | ~df_labeled["사업분류재정구분"].isin(["공통", "자체"])
)
structure_excluded = df_labeled[leaf_candidate_mask & structure_invalid_mask].copy()

print("구조 이상 제외 행 수:", len(structure_excluded))
display(
    structure_excluded[["지역", "원본행", "세부사업명", "사업분류재정구분", "대분류", "중분류"]]
)

# 구조 이상 행을 제외한 최종 leaf를 추출한다.
df_leaf = df_labeled[leaf_candidate_mask & ~structure_invalid_mask].copy()

candidate_leaf_count = len(
    df_leaf[df_leaf["지역"].eq(decision["지역"]) & df_leaf["원본행"].eq(decision["후보_원본행"])]
)
merged_leaf = df_leaf[
    df_leaf["지역"].eq(decision["지역"]) & df_leaf["원본행"].eq(decision["병합대상_원본행"])
]
assert candidate_leaf_count == 0
assert len(merged_leaf) == 1
assert merged_leaf.iloc[0]["세부사업명"] == "학대피해아동 보호를 위한 안전망 구축"
pd.testing.assert_series_equal(
    merged_leaf.iloc[0][preserved_target_cols],
    target_values_before,
    check_names=False,
)

assert len(structure_excluded) == 7
assert len(df_leaf) == 8122
print("후보 판정 후 최종 leaf shape:", df_leaf.shape)
print(df_leaf.columns.tolist())
df_leaf.head()

구조 이상 제외 행 수: 7


,지역,원본행,세부사업명,사업분류재정구분,대분류,중분류
1312,인천,1505,합계(공통사업+자체사업),NaN,NaN,NaN
2265,울산,2597,(울산광역시･울산광역시교육청),(울산광역시･울산광역시교육청),NaN,NaN
2656,경기,3044,(경기도･경기도교육청),(경기도･경기도교육청),NaN,NaN
4546,충북,5199,(충청북도･충청북도교육청),(충청북도･충청북도교육청),NaN,NaN
5060,충남,5792,(충청남도･충청남도교육청),(충청남도･충청남도교육청),NaN,NaN
5705,전북,6521,(전라북도･전라북도교육청),(전라북도･전라북도교육청),NaN,NaN
6224,전남,7112,(전라남도･전라남도교육청),(전라남도･전라남도교육청),NaN,NaN


후보 판정 후 최종 leaf shape: (8122, 16)
['세부사업명', '사업분류재정구분', '2022년 예산', '2021년 예산', '증감액', '비율', '주요내용', '원본표구간', '머리글행', '원본행', '행유형', '사업행구분', '후보판정_제외', '대분류', '중분류', '지역']


,세부사업명,사업분류재정구분,2022년 예산,2021년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,후보판정_제외,대분류,중분류,지역
2,저출생 인식개선 캠페인,공통,58.0,43,15,34.9,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",1,2,6,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
3,어린이 안전 영상정보 인프라 구축,공통,38039.0,33280,4759,14.3,지원대상 : 어린이\n지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,1,2,7,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
4,입양아동 가족지원,공통,4780.0,4120,660,16,"지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정\n지원내용 : 입양아동양육보조금, 심리치료비 등",1,2,8,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
5,국공립어린이집 등 보육서비스 수준제고,공통,330577.0,322975,7602,2.4,지원대상 : 국공립어린이집 등\n지원내용 : 보육교직원 인건비 지원,1,2,9,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
6,국공립어린이집 확충,공통,15523.0,40000,-24477,-61.2,국공립어린이집 이용률 46%달성,1,2,10,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울


In [10]:
mask_non_numeric = (
    to_numeric_budget(df_leaf[CURRENT_BUDGET_COL]).isna() & df_leaf[CURRENT_BUDGET_COL].notna()
)
display(df_leaf.loc[mask_non_numeric, ["지역", "세부사업명", CURRENT_BUDGET_COL]])

,지역,세부사업명,2022년 예산


In [11]:
# QA: 중분류 소계행과 세부사업 합계를 공통 기준으로 대조한다.
from src.features.pipeline_common import build_subtotal_qa

# 병합 후보행과 구조 이상 7행은 최종 leaf와 동일하게 QA 합계에서도 제외한다.
df_qa_labeled = df_labeled.copy()
final_leaf_mask = leaf_candidate_mask & ~structure_invalid_mask
qa_excluded_mask = df_labeled["사업행구분"].eq("세부사업") & ~final_leaf_mask
df_qa_labeled.loc[qa_excluded_mask, "사업행구분"] = "구조이상_제외"

qa = build_subtotal_qa(
    df_qa_labeled,
    budget_col=CURRENT_BUDGET_COL,
    tolerance=0.01,
    rate_tolerance=10.0,
)

# 원본 대조용 행 번호를 기존 진단 로직에 유지한다.
subtotal_rows = df_labeled.loc[
    df_labeled["사업행구분"].eq("중분류_소계"),
    ["지역", "대분류", "중분류", "원본행"],
]
qa = qa.merge(
    subtotal_rows,
    on=["지역", "대분류", "중분류"],
    how="left",
    validate="one_to_one",
)
qa["원시값_완전동일_결과"] = np.where(qa["leaf_합계"] == qa["원본_소계값"], "일치", "불일치")
qa["원시값_완전동일_결과"].value_counts()

원시값_완전동일_결과
일치     118
불일치     16
Name: count, dtype: int64

In [12]:
qa.head()

,지역,대분류,중분류,원본_소계값,leaf_합계,차이,오차율(%),QA_병합상태,결과,허용기준결과,원본행,원시값_완전동일_결과
0,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),464214.8,464214.9,0.1,0.0,양쪽존재,불일치,허용,4586,불일치
1,강원,Ⅰ. 공통사업,2. 건강하고 능동적인 고령사회(공통),323741.0,323741.0,0.0,0.0,양쪽존재,일치,허용,4651,일치
2,강원,Ⅰ. 공통사업,3. 모두의 역량이 고루\n발휘되는 사회(공통),82018.0,82018.0,0.0,0.0,양쪽존재,일치,허용,4679,일치
3,강원,Ⅰ. 공통사업,4.인구구조 변화에 대한\n적응(공통),75598.3,75598.3,0.0,0.0,양쪽존재,일치,허용,4691,일치
4,강원,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께\n돌보는 사회(자체),251476.8,251476.8,0.0,0.0,양쪽존재,일치,허용,4718,일치


In [13]:
# 부동소수점 표현 노이즈를 제거하기 위해 차이는 소수점 첫째 자리로 고정한다.
qa["차이"] = (qa["leaf_합계"] - qa["원본_소계값"]).round(1)

# 원시값 완전 동일과 소수점 첫째 자리 반올림 후 차이 0.0을 별도 컬럼으로 유지한다.
qa["소수점첫째자리_반올림후_차이0.0_결과"] = qa["차이"].eq(0.0).map({True: "일치", False: "불일치"})

qa["검토구분"] = np.select(
    [
        qa["소수점첫째자리_반올림후_차이0.0_결과"].eq("일치"),
        qa["차이"].abs().le(0.5),
    ],
    ["일치", "미세 차이"],
    default="실질 검토",
)
qa_mismatch = qa[qa["소수점첫째자리_반올림후_차이0.0_결과"] == "불일치"].copy()
qa_micro_mismatch = qa_mismatch[qa_mismatch["차이"].abs().le(0.5)].copy()
qa_material = qa_mismatch[qa_mismatch["차이"].abs().gt(0.5)].copy()
assert len(qa_mismatch) == len(qa_micro_mismatch) + len(qa_material)

# #15 기준: 원본 소계 대비 절대 오차율 10%를 초과하거나 판정할 수 없는 항목만 검토한다.
qa_review = qa.loc[qa["허용기준결과"].ne("허용")].sort_values(
    "오차율(%)",
    key=lambda x: x.abs(),
    ascending=False,
    na_position="last",
)

exact_matches = int(qa["원시값_완전동일_결과"].eq("일치").sum())
rounded_matches = int(qa["소수점첫째자리_반올림후_차이0.0_결과"].eq("일치").sum())
assert (exact_matches, len(qa) - exact_matches) == (118, 16)
assert (rounded_matches, len(qa) - rounded_matches) == (119, 15)
assert (len(qa_micro_mismatch), len(qa_material)) == (12, 3)
assert qa["허용기준결과"].value_counts().to_dict() == {"허용": 134}
assert len(qa_review) == 0
print(f"원시값 완전 동일: 일치 {exact_matches}건 / 불일치 {len(qa) - exact_matches}건")
print(
    "소수점 첫째 자리 반올림 후 차이 0.0: "
    f"일치 {rounded_matches}건 / 불일치 {len(qa) - rounded_matches}건"
)
print(
    f"반올림 후 불일치 분류: 미세 차이 {len(qa_micro_mismatch)}건 / 실질 검토 {len(qa_material)}건"
)
print("10% 오차율 허용기준:", qa["허용기준결과"].value_counts().to_dict())

원시값 완전 동일: 일치 118건 / 불일치 16건
소수점 첫째 자리 반올림 후 차이 0.0: 일치 119건 / 불일치 15건
반올림 후 불일치 분류: 미세 차이 12건 / 실질 검토 3건
10% 오차율 허용기준: {'허용': 134}


In [14]:
# 기본 QA 출력은 10% 허용범위를 초과했거나 판정불가인 항목만 보여준다.
display(
    qa_review[
        [
            "지역",
            "대분류",
            "중분류",
            "원본행",
            "원본_소계값",
            "leaf_합계",
            "차이",
            "오차율(%)",
            "QA_병합상태",
            "결과",
            "허용기준결과",
        ]
    ]
)
print("허용기준 외 검토 대상:", len(qa_review), "건")

,지역,대분류,중분류,원본행,원본_소계값,leaf_합계,차이,오차율(%),QA_병합상태,결과,허용기준결과


허용기준 외 검토 대상: 0 건


# 2022 QA 실질 불일치 원인 확인 - 경북·경남 원본 대조

`차이 = leaf_합계 - 원본_소계값`을 소수점 첫째 자리로 반올림한 뒤, `abs(차이) > 0.5`인 실질 검토 대상 3건을 진단한다.

- 경북 `1. 함께 일하고 함께 돌보는 사회(자체)`: -172.3
- 경남 `3. 모두의 역량이 고루 발휘되는 사회(공통)`: -1391.0
- 경남 `4.인구구조 변화에 대한 적응(자체)`: +52.0

`원본행`은 `Table 1`의 1-indexed 엑셀 행 번호이므로, 소계 위치와 원본행 간격을 함께 출력해 입력자료 이상과 파싱 문제를 구분한다.


In [15]:
expected_material = {
    ("경북", -172.3),
    ("경남", -1391.0),
    ("경남", 52.0),
}
actual_material = set(zip(qa_material["지역"], qa_material["차이"]))
assert actual_material == expected_material, actual_material

display(qa_material[["지역", "대분류", "중분류", "원본행", "원본_소계값", "leaf_합계", "차이"]])

,지역,대분류,중분류,원본행,원본_소계값,leaf_합계,차이
18,경남,Ⅰ. 공통사업,3. 모두의 역량이 고루 발휘되는 사회(공통),8726,28285.0,26894.0,-1391.0
23,경남,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한 적응(자체),9388,9353.0,9405.0,52.0
28,경북,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),7881,402270.0,402097.7,-172.3


In [16]:
def show_labeled_subtotal_context(sido: str, medium_category: str, window: int = 2):
    """정리본_자동에서 대상 중분류 소계 앞뒤 문맥을 원본행 순서로 표시한다."""
    region_df = df_labeled[df_labeled["지역"].eq(sido)].sort_values("원본행")
    subtotal_rows = region_df[
        region_df["사업행구분"].eq("중분류_소계") & region_df["세부사업명"].eq(medium_category)
    ]
    assert len(subtotal_rows) == 1, (sido, medium_category, len(subtotal_rows))
    center_pos = region_df.index.get_loc(subtotal_rows.index[0])
    view = region_df.iloc[max(0, center_pos - window) : center_pos + window + 1]
    print(f"--- {sido} / {medium_category} / 정리본_자동 소계 문맥 ---")
    display(view[["원본행", "세부사업명", "사업행구분", "대분류", "중분류", CURRENT_BUDGET_COL]])


for target in qa_material.itertuples(index=False):
    show_labeled_subtotal_context(target.지역, target.중분류)

--- 경남 / 3. 모두의 역량이 고루 발휘되는 사회(공통) / 정리본_자동 소계 문맥 ---


,원본행,세부사업명,사업행구분,대분류,중분류,2022년 예산
7635,8724,노인요양시설확충\n(기능보강) 사업,세부사업,Ⅰ. 공통사업,2. 건강하고 능동적인 고령사회(공통),7784.0
7636,8725,공설 장사시설 설치,세부사업,Ⅰ. 공통사업,2. 건강하고 능동적인 고령사회(공통),6751.0
7637,8726,3. 모두의 역량이 고루 발휘되는 사회(공통),중분류_소계,Ⅰ. 공통사업,3. 모두의 역량이 고루 발휘되는 사회(공통),28285.0
7638,8727,지역혁신 플랫폼 구축,세부사업,Ⅰ. 공통사업,3. 모두의 역량이 고루 발휘되는 사회(공통),0.0
7639,8728,경남 사회적경제 청년부흥\n프로젝트,세부사업,Ⅰ. 공통사업,3. 모두의 역량이 고루 발휘되는 사회(공통),1268.0


--- 경남 / 4.인구구조 변화에 대한 적응(자체) / 정리본_자동 소계 문맥 ---


,원본행,세부사업명,사업행구분,대분류,중분류,2022년 예산
8213,9386,4-16.행복교육지구운영,세부사업,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루\n발휘되는 사회(자체),300.0
8214,9387,종합복지관 취미건강프로그램 운영,세부사업,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루\n발휘되는 사회(자체),119.0
8215,9388,4.인구구조 변화에 대한 적응(자체),중분류_소계,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한 적응(자체),9353.0
8216,9389,3-7.여성결혼이민자 (다문화가족 한마음대회),세부사업,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한 적응(자체),40.0
8217,9390,3-8.여성결혼이민자\n(국적 및 자격취득비),세부사업,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한 적응(자체),12.0


--- 경북 / 1. 함께 일하고 함께 돌보는 사회(자체) / 정리본_자동 소계 문맥 ---


,원본행,세부사업명,사업행구분,대분류,중분류,2022년 예산
6898,7879,지역저출산극복네트워크\n지원사업,세부사업,Ⅰ. 공통사업,4. 인구구조 변화에 대한 적응(공통),39.0
6899,7880,Ⅱ. 지자체 자체사업,대분류_소계,Ⅱ. 지자체 자체사업,4. 인구구조 변화에 대한 적응(공통),550192.0
6900,7881,1. 함께 일하고 함께 돌보는 사회(자체),중분류_소계,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),402270.0
6901,7882,다둥이가족 대잔치,세부사업,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),50.0
6902,7883,아이사랑 가족대축제,세부사업,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),48.0


In [17]:
gap_diagnostics = []
for target in qa_material.itertuples(index=False):
    category_rows = df_labeled[
        df_labeled["지역"].eq(target.지역)
        & df_labeled["중분류"].eq(target.중분류)
        & df_labeled["사업행구분"].isin(["중분류_소계", "세부사업"])
    ].sort_values("원본행")
    gaps = category_rows[["원본행", "세부사업명", "사업행구분"]].copy()
    gaps["이전_원본행"] = gaps["원본행"].shift()
    gaps["이전_세부사업명"] = gaps["세부사업명"].shift()
    gaps["원본행_간격"] = gaps["원본행"] - gaps["이전_원본행"]
    gaps = gaps[gaps["원본행_간격"].gt(1)].copy()
    gaps.insert(0, "중분류", target.중분류)
    gaps.insert(0, "지역", target.지역)
    gap_diagnostics.append(gaps)

qa_source_gaps = pd.concat(gap_diagnostics, ignore_index=True)
display(qa_source_gaps)

,지역,중분류,원본행,세부사업명,사업행구분,이전_원본행,이전_세부사업명,원본행_간격
0,경남,3. 모두의 역량이 고루 발휘되는 사회(공통),8742,경남혁신도시 지역인재 채용 확대,세부사업,8738.0,지역사회서비스\n청년사업단 운영,4.0
1,경남,4.인구구조 변화에 대한 적응(자체),9407,입양가정 지원(도),세부사업,9403.0,결혼이민자 영유아기 자녀\n양육서비스,4.0
2,경남,4.인구구조 변화에 대한 적응(자체),9435,김해시 안전도시연구센터\n설치 운영,세부사업,9429.0,샛별기업(G-Rising)기업 육성사업,6.0
3,경남,4.인구구조 변화에 대한 적응(자체),9464,4-6.남해대학 귀농귀촌 정규교육과정 운영,세부사업,9460.0,4-4.전입세대 공영주차장 이용권 제공,4.0
4,경북,1. 함께 일하고 함께 돌보는 사회(자체),7905,인성교육 프로그램 지원사업,세부사업,7899.0,육아종합지원센터 프로그램 운영,6.0
5,경북,1. 함께 일하고 함께 돌보는 사회(자체),7929,청소년 부모교육 운영,세부사업,7925.0,입양아동상해보험료 지원,4.0
6,경북,1. 함께 일하고 함께 돌보는 사회(자체),7959,장애유아 의무교육비\n지원,세부사업,7953.0,농산어촌 방과후학교 운영 지원,6.0
7,경북,1. 함께 일하고 함께 돌보는 사회(자체),7978,어린이집보육교사복지증진,세부사업,7974.0,정부미지원어린이집\n보육교사처우개선비,4.0
8,경북,1. 함께 일하고 함께 돌보는 사회(자체),8003,임산부의 날 기념 아카데미 운영,세부사업,7997.0,임신축하금 지원,6.0
9,경북,1. 함께 일하고 함께 돌보는 사회(자체),8026,출산장려금 지원,세부사업,8022.0,출산축하금 지원,4.0


In [18]:
def show_table1_around(center_excel_row: int, window: int, label: str):
    """Table 1 원본 시트에서 특정 엑셀 행(1-indexed) 주변의 비어 있지 않은 열을 표시한다."""
    start = max(1, center_excel_row - window)
    end = min(len(df_table1), center_excel_row + window)
    print(f"--- {label} (Table 1 엑셀행 {start}~{end}) ---")
    view = df_table1.iloc[start - 1 : end].copy()
    view.insert(0, "Table1_엑셀행", range(start, end + 1))
    view = view.dropna(axis=1, how="all")
    display(view)


region_name_map = {"경북": "경상북도", "경남": "경상남도"}


def show_table1_subtotal_context(sido: str, medium_category: str, window: int = 3):
    """해당 시도 원문 구간에서 중분류 소계를 찾아 Table 1 문맥을 표시한다."""
    full_region_name = region_name_map[sido]
    start_pos = next(i for i, (_, name) in enumerate(region_starts) if name == full_region_name)
    start_row = region_starts[start_pos][0]
    end_row = (
        region_starts[start_pos + 1][0] if start_pos + 1 < len(region_starts) else len(df_table1)
    )
    normalized_title = re.sub(r"\s+", "", medium_category)
    region_text = table1_row_text.iloc[start_row:end_row]
    matches = region_text[region_text.str.contains(re.escape(normalized_title), regex=True)]
    assert len(matches) == 1, (sido, medium_category, list(matches.index + 1))
    show_table1_around(
        int(matches.index[0]) + 1,
        window,
        f"{sido} / {medium_category}",
    )


for target in qa_material.itertuples(index=False):
    show_table1_subtotal_context(target.지역, target.중분류)

print("원본행 간격 후보는 위 qa_source_gaps와 Table 1 원문을 함께 대조해 원인을 판정한다.")

--- 경남 / 3. 모두의 역량이 고루 발휘되는 사회(공통) (Table 1 엑셀행 8723~8729) ---


,Table1_엑셀행,0,2,4,8,12,17,22
8722,8723,양로시설 운영비 지원,공통,5148,5470,-322,-5.9,• 양로시설 운영비 및 종사자 인건비 지원
8723,8724,노인요양시설확충\n(기능보강) 사업,공통,7784,13263,-5479,-41.3,"• 노인요양시설 등 신축, 증축, 개보수, 장비보강"
8724,8725,공설 장사시설 설치,공통,6751,9847,-3096,-31.4,"• 공설 장사시설(화장장, 봉안당 등) 설치지원"
8725,8726,3. 모두의 역량이 고루 발휘되는 사회(공통),<NA>,28285,96739,-68454,-70.8,<NA>
8726,8727,지역혁신 플랫폼 구축,공통,-,66000,-66000,-100,• 경남형 공유대학(USG) 중심 혁신인재 양성\n• 지역산업 활력회복 연계 추진(3대 핵심분야)
8727,8728,경남 사회적경제 청년부흥\n프로젝트,공통,1268,3291,-631,-19.2,• 사회적경제기업 취업청년 인건비 등 지원
8728,8729,대학일자리센터,공통,1900,2700,-800,-29.6,"• 대학 및 청년에 대한 원스톱 취업지원 서비스 제공 (진로지도, 취･창업 알선 등)"


--- 경남 / 4.인구구조 변화에 대한 적응(자체) (Table 1 엑셀행 9385~9391) ---


,Table1_엑셀행,0,2,4,8,12,17,22
9384,9385,4-15.수도권남해학숙설치,자체,25,25,-,-,• 사업대상 : 수도권 대학에 진학 남해출신 대학생\n• 지원내용 : 학숙 운영비 지원
9385,9386,4-16.행복교육지구운영,자체,300,300,-,-,"• 사업대상 : 청소년\n• 사업내용 : 지자체와 교육청 공동사업, 생활터전, 학교,\n바다마을학교 등 운영"
9386,9387,종합복지관 취미건강프로그램 운영,자체,119,74,45,60.8,• 성인대상 취미교실 운영
9387,9388,4.인구구조 변화에 대한 적응(자체),<NA>,9353,9860,284,161.4,<NA>
9388,9389,3-7.여성결혼이민자 (다문화가족 한마음대회),자체,40,8,32,400,• 다문화가족 대상 한마음대회 개최(1회)
9389,9390,3-8.여성결혼이민자\n(국적 및 자격취득비),자체,12,12,-,-,• 결혼이민여성 대상 국적 및 자격취득비 지원
9390,9391,인구감소 극복과\n인구유입을 위한 공모사업,자체,800,810,-10,-1.2,• 지원대상 : 도내 시군(2~3개소)\n• 사업내용 : 인구감소 극복 공모사업비 지원


--- 경북 / 1. 함께 일하고 함께 돌보는 사회(자체) (Table 1 엑셀행 7878~7884) ---


,Table1_엑셀행,1,6,10,14,19,25,27
7877,7878,4. 인구구조 변화에 대한 적응(공통),<NA>,39,39,0,0,<NA>
7878,7879,지역저출산극복네트워크\n지원사업,공통,39,39,0,28,"• 경상북도 저출산극복 사회연대회의 운영\n• 다양한 홍보･캠페인 실시, 사회운동 전개 등"
7879,7880,Ⅱ. 지자체 자체사업,·,550192,491822.7,58369.3,11.9,<NA>
7880,7881,1. 함께 일하고 함께 돌보는 사회(자체),<NA>,402270,339736,62361.7,18.4,<NA>
7881,7882,다둥이가족 대잔치,자체,50,50,0,0,"• 도내 다둥이 가정을 대상으로 가족공연, 무대행사, 체험부스 운영 등"
7882,7883,아이사랑 가족대축제,자체,48,40,8,20,"• 지역 임산부, 초등학교 이하 자녀･부모, 예비부부\n등이 직접 참여하는 체험프로그램 행사 추진"
7883,7884,출산장려 캠페인,자체,200,200,0,0,• 언론매체를 통한 저출산 극복 인식개선 및\n출산정책 홍보 등


원본행 간격 후보는 위 qa_source_gaps와 Table 1 원문을 함께 대조해 원인을 판정한다.


# 2022년 증감액 부호 검증

2022년 원본 `증감액`의 부호가 `2022년 예산 - 2021년 예산`과 일치하는지 전국 17개 시도에서 확인한다.

당해·전년도 예산과 원본 증감액은 모두 상단의 `to_numeric_budget`로 변환하며, 최종 증감액·증감율은 당해·전년도 예산으로 직접 재계산한다.


In [19]:
# 상단의 공통 숫자 변환 함수를 당해·전년도 예산과 원본 증감액에 재사용한다.
df_raw[f"{YEAR}년 예산_num"] = to_numeric_budget(df_raw[CURRENT_BUDGET_COL])
df_raw[f"{PREVIOUS_YEAR}년 예산_num"] = to_numeric_budget(df_raw[PREVIOUS_BUDGET_COL])
df_raw["증감액_num"] = to_numeric_budget(df_raw["증감액"])
df_raw["계산된_증감액"] = df_raw[f"{YEAR}년 예산_num"] - df_raw[f"{PREVIOUS_YEAR}년 예산_num"]

# 실제로 예산이 감소한 행(계산된 증감액 < 0)만 골라서, 증감액 컬럼 부호를 대조
감소행 = df_raw[df_raw["계산된_증감액"] < -0.5].copy()
감소행["부호_소실"] = 감소행["증감액_num"] > 0.5

print("예산 감소 행 수:", len(감소행))
print("그중 증감액이 양수로 찍힌(부호 소실) 행 수:", 감소행["부호_소실"].sum())

예산 감소 행 수: 1510
그중 증감액이 양수로 찍힌(부호 소실) 행 수: 4


In [20]:
# 지역별로 부호 소실 비율 확인
지역별_부호소실_비율 = (
    감소행.groupby("지역")["부호_소실"].mean().mul(100).round(1).sort_values(ascending=False)
)
지역별_부호소실_비율

지역
대전    1.7
경남    1.2
경북    0.9
강원    0.0
울산    0.0
충남    0.0
제주    0.0
전북    0.0
전남    0.0
인천    0.0
서울    0.0
세종    0.0
경기    0.0
부산    0.0
대구    0.0
광주    0.0
충북    0.0
Name: 부호_소실, dtype: float64

# 증감액 / 비율 직접 재계산

- 위에서 확인했듯이 원본에서는 시도별로 부호나 값 자체가 틀린 경우가 있어 신뢰할 수 없음.
- 따라서 직접 재계산


In [21]:
# 상단의 공통 숫자 변환 함수를 leaf 예산 재계산에도 재사용한다.
current_num_col = f"{YEAR}년_예산_num"
previous_num_col = f"{PREVIOUS_YEAR}년_예산_num"
df_leaf[previous_num_col] = to_numeric_budget(df_leaf[PREVIOUS_BUDGET_COL])
df_leaf[current_num_col] = to_numeric_budget(df_leaf[CURRENT_BUDGET_COL])
df_leaf["증감액_재계산"] = df_leaf[current_num_col] - df_leaf[previous_num_col]
df_leaf["증감율_재계산"] = (
    df_leaf["증감액_재계산"] / df_leaf[previous_num_col].replace(0, np.nan) * 100
).round(1)

df_leaf[["세부사업명", current_num_col, previous_num_col, "증감액_재계산", "증감율_재계산"]].head(
    10
)

,세부사업명,2022년_예산_num,2021년_예산_num,증감액_재계산,증감율_재계산
2,저출생 인식개선 캠페인,58.0,43.0,15.0,34.9
3,어린이 안전 영상정보 인프라 구축,38039.0,33280.0,4759.0,14.3
4,입양아동 가족지원,4780.0,4120.0,660.0,16.0
5,국공립어린이집 등 보육서비스 수준제고,330577.0,322975.0,7602.0,2.4
6,국공립어린이집 확충,15523.0,40000.0,-24477.0,-61.2
7,어린이집 이용 영유아 무상보육 확대,791160.0,854364.0,-63204.0,-7.4
8,육아종합지원센터 운영,6293.0,6104.0,189.0,3.1
9,가정양육수당 지원,115422.0,171371.0,-55949.0,-32.6
10,부모 모니터링단 운영,242.0,272.0,-30.0,-11.0
11,공동육아나눔터,2030.0,1790.0,240.0,13.4


# 최종 스키마


In [22]:
# 텍스트 정리
PUA_PATTERN = re.compile(r"[\uE000-\uF8FF]")


def clean_text(series: pd.Series) -> pd.Series:
    """줄바꿈, 공백을 단일 공백으로 정리"""

    def _clean(x):
        if pd.isna(x):
            return x
        x = PUA_PATTERN.sub("", str(x))
        return re.sub(r"\s+", " ", x).strip()

    return series.apply(_clean)


df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"])
df_leaf["대분류"] = clean_text(df_leaf["대분류"])
df_leaf["중분류"] = clean_text(df_leaf["중분류"])

df_leaf.head()

,세부사업명,사업분류재정구분,2022년 예산,2021년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,후보판정_제외,대분류,중분류,지역,2021년_예산_num,2022년_예산_num,증감액_재계산,증감율_재계산
2,저출생 인식개선 캠페인,공통,58.0,43,15,34.9,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",1,2,6,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,43.0,58.0,15.0,34.9
3,어린이 안전 영상정보 인프라 구축,공통,38039.0,33280,4759,14.3,지원대상 : 어린이 지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,1,2,7,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,33280.0,38039.0,4759.0,14.3
4,입양아동 가족지원,공통,4780.0,4120,660,16,"지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정 지원내용 : 입양아동양육보조금, 심리치료비 등",1,2,8,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,4120.0,4780.0,660.0,16.0
5,국공립어린이집 등 보육서비스 수준제고,공통,330577.0,322975,7602,2.4,지원대상 : 국공립어린이집 등 지원내용 : 보육교직원 인건비 지원,1,2,9,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,322975.0,330577.0,7602.0,2.4
6,국공립어린이집 확충,공통,15523.0,40000,-24477,-61.2,국공립어린이집 이용률 46%달성,1,2,10,데이터행,세부사업,False,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,40000.0,15523.0,-24477.0,-61.2


In [23]:
# 컬럼명 정리 + 연도 추가 + 최종 스키마만 선택
df_leaf_final = (
    df_leaf.drop(columns=["증감액", "비율"])
    .rename(
        columns={
            current_num_col: "당해예산",
            previous_num_col: "전년도예산",
            "증감액_재계산": "증감액",
            "증감율_재계산": "증감율",
        }
    )
    .assign(연도=YEAR)
)

df_leaf_final = df_leaf_final[
    [
        "연도",
        "지역",
        "대분류",
        "중분류",
        "사업분류재정구분",
        "세부사업명",
        "주요내용",
        "당해예산",
        "전년도예산",
        "증감액",
        "증감율",
        "원본행",
    ]
]

assert len(df_leaf_final) == 8122
print(df_leaf_final.shape)
display(df_leaf_final.head())

(8122, 12)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,당해예산,전년도예산,증감액,증감율,원본행
2,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,저출생 인식개선 캠페인,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",58.0,43.0,15.0,34.9,6
3,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,어린이 안전 영상정보 인프라 구축,지원대상 : 어린이 지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,38039.0,33280.0,4759.0,14.3,7
4,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,입양아동 가족지원,"지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정 지원내용 : 입양아동양육보조금, 심리치료비 등",4780.0,4120.0,660.0,16.0,8
5,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 등 보육서비스 수준제고,지원대상 : 국공립어린이집 등 지원내용 : 보육교직원 인건비 지원,330577.0,322975.0,7602.0,2.4,9
6,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 확충,국공립어린이집 이용률 46%달성,15523.0,40000.0,-24477.0,-61.2,10


In [24]:
for sido in df_leaf_final["지역"].unique():
    sido_labeled = df_labeled[df_labeled["지역"] == sido]

    sido_dir = get_sido_dir(sido)

    sido_labeled.to_csv(
        sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv", index=False, encoding="utf-8-sig"
    )

# QA 결과는 전체 17개 시도 한 파일로 저장한다(원시값 완전 동일/첫째 자리 반올림 결과 포함).
qa.to_csv(REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ", df_leaf_final["지역"].nunique(), "개 시도")

저장 완료:  17 개 시도


# 세부사업명 오탈자 / 중복 탐지

- 오탈자를 자동으로 고치지 않고, 사람이 검토할 후보만 찾는다.
- 같은 지역 / 중분류 안에서 완전히 같지는 않지만 유사도가 높은 세부사업명 쌍을 `rapidfuzz`로 찾는다.


In [25]:
from itertools import combinations
from rapidfuzz import fuzz

In [26]:
def normalize_for_match(name: str) -> str:
    """clean_text로 정제된 세부사업명에서, 비교 목적으로 공백, 문장부호 마저 제거"""
    return re.sub(r"[\s,-./\-/()]", "", name)


def find_near_duplicates(df: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    """같은 지역 중분류 안에서 완전 일치는 아니지만 유사도가 높은 세부사업명과 쌍을 찾는다."""
    candidates = []
    for (sido, midium_cat), group in df.groupby(["지역", "중분류"]):
        rows = list(
            group[["세부사업명", "당해예산", "주요내용"]].itertuples(index=False, name=None)
        )
        for (name_a, budget_a, content_a), (name_b, budget_b, content_b) in combinations(rows, 2):
            if name_a == name_b:
                continue
            if normalize_for_match(name_a) == normalize_for_match(name_b):
                continue  # 공백/문장부호 차이 -> 검수 대상 x
            score = fuzz.ratio(name_a, name_b)
            if score >= threshold:
                candidates.append(
                    {
                        "지역": sido,
                        "중분류": midium_cat,
                        "세부사업명1": name_a,
                        "당해예산1": budget_a,
                        "주요내용1": content_a,
                        "세부사업명2": name_b,
                        "당해예산2": budget_b,
                        "주요내용2": content_b,
                        "유사도": score,
                    }
                )
    return pd.DataFrame(candidates).sort_values("유사도", ascending=False, kind="stable")


near_dup = find_near_duplicates(df_leaf_final)
print(len(near_dup), "건 발견")
display(near_dup.head(10))

288 건 발견


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
45,경기,1. 함께 일하고 함께 돌보는 사회(자체),"다자녀 가정, 한부모가정 수도요금 감면",0.0,다자녀･한부모 가정 월 사용량 10㎥ 해당 상수도사용료 감면,"다자녀 가정, 한부모가정 하수도요금 감면",0.0,다자녀･한부모 가정 월 사용량 10㎥ 해당 하수도사용료 감면,97.674419
150,부산,1. 함께 일하고 함께 돌보는 사회(자체),일생활 균형을 위한 사회적 분위기 확산,1.5,"일생활 균형 교육 및 캠페인 실시, 가족사랑의 날 운영",일･생활 균형을 위한 사회적 분위기 확산,1.0,일･생활 균형 사회분위기 조성 캠페인 등,97.674419
130,부산,1. 함께 일하고 함께 돌보는 사회(자체),정부지원 어린이집 차량운영비 지원,171.0,통학버스를 운영하는 정부지원 어린이집에 차량운영비 지원(개소당 월50천원),정부미지원 어린이집 차량운영비 지원,9.0,정부미지원어린이집 차량운영비 연 600천 원 지원,97.297297
131,부산,1. 함께 일하고 함께 돌보는 사회(자체),정부지원 어린이집 차량운영비 지원,171.0,통학버스를 운영하는 정부지원 어린이집에 차량운영비 지원(개소당 월50천원),정부미지원 어린이집 차량운영비 지원,5.0,"차량운영 중인 정부미지원(민간, 가정)어린이집 8개소 대상으로 차량 운영비(유류비) 연 지원",97.297297
133,부산,1. 함께 일하고 함께 돌보는 사회(자체),정부지원 어린이집 차량운영비 지원,171.0,통학버스를 운영하는 정부지원 어린이집에 차량운영비 지원(개소당 월50천원),정부미지원 어린이집 차량운영비 지원,72.0,부산광역시 사하구 소재 정부미지원 평가인증 어린이집에 차량운영비 지원,97.297297
2,강원,2. 건강하고 능동적인 고령사회 구축(자체),저소득 노인세대 건강보험료 지원,77.0,"저소득 노인가구 건강보험료, 장기요양보험료 지원",저소득층 노인세대 건강보험료 지원,139.0,영월군에 주소를 둔 국민건강보험공단 지역가입자로서 건강보험료 부과금액이 ‘최저보험료 이하’인 65세 이상 노인세대,97.142857
61,경기,1. 함께 일하고 함께 돌보는 사회(자체),출산가정 수도요금 감면(신규),0.0,출생신고 한세대 36개월간 수도요금 감면,출산가정 하수도요금 감면(신규),0.0,출생신고 한세대 36개월간 하수도요금 감면,96.969697
101,경북,1. 함께 일하고 함께 돌보는 사회(자체),임산부 태아 기형아 검사비 지원,24.0,"• 지원대상 : 임신10주~20주 임산부 (1차 임신10주~14주, 2차 임신14주~20주) • 지원내용 : 회차당 쿠폰 35,000원 지원",임부 태아 기형아 검사비 지원,16.0,• 임부 태아 기형아검사비 지원,96.969697
125,광주,2. 건강하고 능동적인 고령사회 구축(자체),노인장기요양보험 등급자 지원,48000.0,"저소득층 노인에게 시설급여, 재가급여 지원",노인장기요양보험 등급외자 지원,222.0,기초수급자 및 실비 입소 등급외자에 대한 시설 이용 비용 지원,96.774194
201,부산,2. 건강하고 능동적인 고령사회 구축(자체),저소득 노인 건강보험료 지원,40.0,저소득 노인세대 국민건강보험료 지원,저소득층 노인 건강보험료 지원,65.0,보건복지부 장관이 정하여 고시하는 최저보험료 이하 납부하는 65세 이상 노인세대의 건강보험료 지원,96.774194


In [27]:
# 당해예산이 완전히 같은 쌍은 진짜 같은 사업일 가능성이 높은 후보
near_dup_same_budget = near_dup[near_dup["당해예산1"] == near_dup["당해예산2"]]

print(len(near_dup_same_budget), "건 (전체", len(near_dup), "건 중)")
display(near_dup_same_budget)

7 건 (전체 288 건 중)


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
45,경기,1. 함께 일하고 함께 돌보는 사회(자체),"다자녀 가정, 한부모가정 수도요금 감면",0.0,다자녀･한부모 가정 월 사용량 10㎥ 해당 상수도사용료 감면,"다자녀 가정, 한부모가정 하수도요금 감면",0.0,다자녀･한부모 가정 월 사용량 10㎥ 해당 하수도사용료 감면,97.674419
61,경기,1. 함께 일하고 함께 돌보는 사회(자체),출산가정 수도요금 감면(신규),0.0,출생신고 한세대 36개월간 수도요금 감면,출산가정 하수도요금 감면(신규),0.0,출생신고 한세대 36개월간 하수도요금 감면,96.969697
190,부산,1. 함께 일하고 함께 돌보는 사회(자체),어린이집 난방비 지원,74.0,지원대상 : 친환경쌀 구입 어린이집 지원금액 : 어린이집 현원별 차등지원,어린이집 냉난방비 지원,74.0,관내 전체 어린이집 냉난방비 지원,95.652174
62,경기,1. 함께 일하고 함께 돌보는 사회(자체),다자녀(세자녀이상)가구 상수도 요금 감면,0.0,미성년 3자녀를 둔 가구에 상수도요금 감면,다자녀(세자녀이상)가구 하수도 요금 감면,0.0,미성년 3자녀를 둔 가구에 하수도요금 감면,95.454545
58,경기,1. 함께 일하고 함께 돌보는 사회(자체),다자녀 가정 상수도 요금 감면,0.0,수도요금 감면을 통한 다자녀 가구 지원,다자녀 가정 하수도 요금 감면,0.0,⌜주민등록법⌟상 18세 이하의 자녀가 3명이상 동일세대로 구성되어 있는 가구 등에 대한 하수도 요금감면,93.750000
118,경북,3. 모두의 역량이 고루 발휘되는 사회(자체),중학생 영어체험학습 지원,89.0,• 원어민 영어마을 체험 기회 제공,중학생 영어체험학습 지원사업,89.0,• 중학생 영어마을체험 지원,92.857143
156,부산,1. 함께 일하고 함께 돌보는 사회(자체),유모차 살균소독기 설치 및 운영,0.0,유모차 살균소독기 4대 지속 운영,"유모차 살균소독기 설치, 운영",0.0,유모차 살균소독기 2개소 설치 운영,90.909091


In [28]:
# 당해예산 0.0이 우연일치인지 확인
print("당해예산 = 0 으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] == 0).sum())
print("0이 아닌 값으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] != 0).sum())

near_dup_confident = near_dup_same_budget[near_dup_same_budget["당해예산1"] != 0]
display(near_dup_confident)

# 기존 리포트의 스키마·컬럼 순서·정렬(고신뢰 우선, 유사도 내림차순)을 유지한다.
near_dup_report = (
    near_dup.assign(
        유사도=near_dup["유사도"].round(1),
        예산일치_신뢰도높음=(
            near_dup["당해예산1"].eq(near_dup["당해예산2"]) & near_dup["당해예산1"].ne(0)
        ),
    )
    .rename(
        columns={
            "세부사업명1": "세부사업명_A",
            "당해예산1": "예산_A",
            "세부사업명2": "세부사업명_B",
            "당해예산2": "예산_B",
        }
    )[
        [
            "지역",
            "중분류",
            "유사도",
            "세부사업명_A",
            "예산_A",
            "세부사업명_B",
            "예산_B",
            "예산일치_신뢰도높음",
        ]
    ]
    .sort_values(
        ["예산일치_신뢰도높음", "유사도"],
        ascending=[False, False],
        kind="stable",
    )
    .reset_index(drop=True)
)
near_dup_report.to_csv(
    REPORTS_DIR / f"{YEAR}_세부사업명_오탈자중복_후보.csv",
    index=False,
    encoding="utf-8-sig",
)
print(
    "near duplicate 저장:",
    len(near_dup_report),
    "건 / 고신뢰",
    int(near_dup_report["예산일치_신뢰도높음"].sum()),
    "건",
)

당해예산 = 0 으로 일치한 건수:  5
0이 아닌 값으로 일치한 건수:  2


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
190,부산,1. 함께 일하고 함께 돌보는 사회(자체),어린이집 난방비 지원,74.0,지원대상 : 친환경쌀 구입 어린이집 지원금액 : 어린이집 현원별 차등지원,어린이집 냉난방비 지원,74.0,관내 전체 어린이집 냉난방비 지원,95.652174
118,경북,3. 모두의 역량이 고루 발휘되는 사회(자체),중학생 영어체험학습 지원,89.0,• 원어민 영어마을 체험 기회 제공,중학생 영어체험학습 지원사업,89.0,• 중학생 영어마을체험 지원,92.857143


near duplicate 저장: 288 건 / 고신뢰 2 건


# 주요내용 구조 패턴 검사

- 원자화를 위해 '지원대상:..', '지원내용: ..' 이 모든 행에 포함되어있는지 확인한다.


In [29]:
def dedup_label(text: str) -> str:
    """지원대상 : 지원대상 : ...  처럼 같은 라벨이 연속으로 중복된 걸 하나로 정리"""
    if pd.isna(text):
        return text
    text = re.sub(
        r"\(\s*(지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\s*\)",
        r"\1 : ",
        text,
    )
    text = re.sub(r"(지원대상\s*[:：]\s*)+", "지원대상 : ", text)
    text = re.sub(r"(지원내용\s*[:：]\s*)+", "지원내용 : ", text)
    text = re.sub(r"(사업대상\s*[:：]\s*)+", "사업대상 : ", text)
    text = re.sub(r"(사업내용\s*[:：]\s*)+", "사업내용 : ", text)
    return text


df_leaf_final["주요내용"] = df_leaf_final["주요내용"].apply(dedup_label)

In [30]:
support_pattern = re.compile(r"^지원대상\s*[:：]\s*(.*?)\s*지원내용\s*[:：]\s*(.*)$", re.DOTALL)


def check_pattern(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern.match(text) else "불일치"


df_leaf_final["주요내용_패턴"] = df_leaf_final["주요내용"].apply(check_pattern)

print(df_leaf_final["주요내용_패턴"].value_counts())
print(df_leaf_final["주요내용_패턴"].value_counts(normalize=True).mul(100).round(1))

주요내용_패턴
불일치    7486
일치      628
결측        8
Name: count, dtype: int64
주요내용_패턴
불일치    92.2
일치      7.7
결측      0.1
Name: proportion, dtype: float64


In [31]:
# 범위 넓혀보기
support_pattern_broad = re.compile(
    r"^(지원대상|사업대상|대상)\s*[:：]?\s*(.*?)\s*(지원내용|사업내용|내용)\s*[:：]?\s*(.*)$",
    re.DOTALL,
)


def check_pattern_broad(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern_broad.match(text) else "불일치"


df_leaf_final["주요내용_패턴_확장"] = df_leaf_final["주요내용"].apply(check_pattern_broad)

print(df_leaf_final["주요내용_패턴_확장"].value_counts())
print(df_leaf_final["주요내용_패턴_확장"].value_counts(normalize=True).mul(100).round(1))

# 여전히 안걸리는 샘플 확인
display(
    df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"][
        ["지역", "세부사업명", "주요내용"]
    ].head(20)
)

주요내용_패턴_확장
불일치    7312
일치      802
결측        8
Name: count, dtype: int64
주요내용_패턴_확장
불일치    90.0
일치      9.9
결측      0.1
Name: proportion, dtype: float64


,지역,세부사업명,주요내용
2,서울,저출생 인식개선 캠페인,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등"
6,서울,국공립어린이집 확충,국공립어린이집 이용률 46%달성
12,서울,아이돌봄서비스 확충 및 내실화,지원내용 : 아이돌봄 서비스 확충
13,서울,서울형 초등돌봄체계 우리동네키움센터 확충,지원내용 : 우리동네키움센터 설치
17,서울,누리과정 지원,"유치원 및 어린이집에 다니는 만3~5세아 (141,523명)의 유아학비, 보육료 지원 1인당 유아학비(공립 8만원, 사립･어린이집 26만원) 방과후과정비(공립 5만원, 사립･어린이집 7만원)"
18,서울,맞춤형 방과후학교운영,"방과후학교 사업 지원 : 초중고 1,231교 지원 방과후학교(돌봄) 지원센터 운영"
47,서울,지매노인에 대한 종합적 관리 및 지원 (치매국가책임제),치매조기검진 및 등록관리 치매치료관리비 지원 인지건강프로그램 운영 기억키움학교(치매환자쉼터) 및 가족카페 운영 치매인식개선 및 자원연계사업 중증치매노인 공공후견인 지원
48,서울,예방접종 관리,만12세이하 어린이 17종 및 고령어르신 등 2종 국가예방접종 백신 및 시행비 지원
49,서울,노인요양시설 인프라 구축,"노인요양시설 이용자 급증에 대비하고, 치매･중풍 등 노인성질환 어르신들이 입소하여 생활할 수 있는 노인요양시설 확충"
52,서울,노인보호전문기관 및 전용쉼터 운영,"노인학대 예방 및 피해자 지원활동 노인학대 상담, 피해신고 접수 및 현장조사 인권 교육, 노인학대 예방을 위한 인식개선 운동, 홍보 등 피해자 의료 및 심리서비스 지원, 쉼터 및 일시보호 시설 연계 등 노인보호전문기관(4개소) 학대피해노인전용쉼터(1개소)"


In [32]:
def extract_via_regex(text: str) -> dict:
    """패턴에 걸리면 그대로 분리, 안 걸리면 None"""
    if pd.isna(text):
        return {"지원대상": None, "지원내용": None}
    m = support_pattern_broad.match(text)
    if m:
        return {"지원대상": m.group(2).strip(), "지원내용": m.group(4).strip()}
    return {"지원대상": None, "지원내용": None}


regex_extracted = pd.json_normalize(df_leaf_final["주요내용"].apply(extract_via_regex))
df_leaf_final["지원대상"] = regex_extracted["지원대상"]
df_leaf_final["지원내용_상세"] = regex_extracted["지원내용"]

In [33]:
df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"].head()

,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,당해예산,전년도예산,증감액,증감율,원본행,주요내용_패턴,주요내용_패턴_확장,지원대상,지원내용_상세
2,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,저출생 인식개선 캠페인,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",58.0,43.0,15.0,34.9,6,불일치,불일치,NaN,NaN
6,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 확충,국공립어린이집 이용률 46%달성,15523.0,40000.0,-24477.0,-61.2,10,불일치,불일치,NaN,NaN
12,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,아이돌봄서비스 확충 및 내실화,지원내용 : 아이돌봄 서비스 확충,47081.0,45658.0,1423.0,3.1,16,불일치,불일치,NaN,NaN
13,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,서울형 초등돌봄체계 우리동네키움센터 확충,지원내용 : 우리동네키움센터 설치,8570.0,10670.0,-2100.0,-19.7,17,불일치,불일치,NaN,NaN
17,2022,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,누리과정 지원,"유치원 및 어린이집에 다니는 만3~5세아 (141,523명)의 유아학비, 보육료 지원 1인당 유아학비(공립 8만원, 사립･어린이집 26만원) 방과후과정비(공립 5만원, 사립･어린이집 7만원)",522375.0,575078.0,-52703.0,-9.2,21,불일치,불일치,NaN,NaN


# 주요내용 LLM 정제 (업스테이지 Solar Pro 3, 구조화 출력)

- LLM은 `주요내용_정제`(오탈자·이상한 공백만 보존형으로 교정) 생성만 담당한다. 요약/재구성 없음, 숫자·고유명사 변경 없음.
- `지원대상`/`지원내용_상세`는 이미 검증된 정규식(`extract_via_regex`) 결과를 그대로 쓴다 — 정규식으로 걸러지는 데 LLM을 또 쓰는 건 실익이 없고 과잉분리 위험만 늘린다.
- `response_format`(json_schema)으로 API 단에서 출력 구조를 강제, 실패 시 1회 재시도 후에도 실패하면 원문을 그대로 사용(정제문장 결측 방지).
- 정제 전후 숫자(금액·비율·연령 등) 보존 여부를 자동 검증해, 달라진 행은 검토 대상으로 표시한다.


In [34]:
import os
import json
import yaml
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

load_dotenv("../.env")

tqdm.pandas()

with open("../configs/llm_extraction.yaml", encoding="utf-8-sig") as f:
    llm_cfg = yaml.safe_load(f)

client = OpenAI(
    api_key=os.environ["UPSTAGE_API_KEY"],
    base_url=llm_cfg["upstage"]["base_url"],
)

RESPONSE_FORMAT = {"type": "json_schema", "json_schema": llm_cfg["response_schema"]}


def call_llm_once(name: str, content: str) -> str | None:
    """API 1회 호출. 파싱 실패하면 None 반환"""
    prompt = llm_cfg["prompt"]["template"].format(name=name, content=content)
    response = client.chat.completions.create(
        model=llm_cfg["upstage"]["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=llm_cfg["upstage"]["temperature"],
        response_format=RESPONSE_FORMAT,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)["정제문장"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


def clean_sentence(name: str, content: str) -> str:
    """주요내용을 LLM으로 정제. 결측이면 결측 유지, 실패 시(1회 재시도 포함) 원문 그대로 사용"""
    if pd.isna(content):
        return None
    for attempt in range(2):  # 최초 시도 + 1회 재시도
        try:
            result = call_llm_once(name, content)
        except Exception as e:
            print(f"API 호출 실패({attempt + 1}회차): {name} -> {e}")
            result = None
        if result is not None:
            return result
    print(f"정제 실패, 원문 유지: {name}")
    return content


def _canonical_numeric_text(raw: str) -> str:
    """표준 천 단위 쉼표만 제거하고 같은 숫자값을 비교 가능한 문자열로 정규화한다."""
    if "," in raw:
        if re.fullmatch(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?", raw) is None:
            return f"invalid-comma:{raw}"
        raw = raw.replace(",", "")

    integer, separator, fraction = raw.partition(".")
    integer = integer.lstrip("0") or "0"
    if separator:
        fraction = fraction.rstrip("0")
        if fraction:
            return f"{integer}.{fraction}"
    return integer


def extract_numbers(text) -> list[tuple[str, str, str]]:
    """부호·소수·퍼센트를 보존하고 표준 천 단위 쉼표만 정규화한 숫자 시퀀스를 반환한다."""
    if pd.isna(text):
        return []

    normalized = str(text).replace("−", "-")
    pattern = re.compile(r"(?P<sign>[+-]?)(?P<number>\d+(?:,\d+)*(?:\.\d+)?)(?P<percent>\s*%)?")
    tokens = []
    for match in pattern.finditer(normalized):
        sign = match.group("sign")
        previous = normalized[: match.start()].rstrip()[-1:]
        if sign and previous and re.fullmatch(r"[\d.,)]", previous):
            sign = ""  # 20-50의 하이픈은 두 번째 숫자의 음수 부호가 아님
        tokens.append(
            (
                sign,
                _canonical_numeric_text(match.group("number")),
                "%" if match.group("percent") else "",
            )
        )
    return tokens


def numbers_preserved(original, cleaned) -> bool:
    """결측 상태와 숫자의 부호·소수·퍼센트 의미가 보존됐는지 확인한다."""
    original_missing = pd.isna(original)
    cleaned_missing = pd.isna(cleaned)
    if original_missing and cleaned_missing:
        return True
    if original_missing or cleaned_missing:
        return False
    return extract_numbers(original) == extract_numbers(cleaned)

# 먼저 소량 샘플로 속도·품질 확인 (전체 실행 전 확인용)

```python
sample = df_leaf_final.sample(
    llm_cfg["sample"]["size"], random_state=llm_cfg["sample"]["random_state"]
).copy()

sample["주요내용_정제"] = sample.progress_apply(
    lambda row: clean_sentence(row["세부사업명"], row["주요내용"]), axis=1
)
sample["숫자보존"] = sample.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~sample["숫자보존"]).sum())
display(
    sample[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세", "숫자보존"]]
)
```


샘플 결과가 괜찮으면 아래 셀로 전체 실행 (7천여 건, 시간 오래 걸릴 수 있음)


In [35]:
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"

PUA_LOW, PUA_HIGH = chr(0xE000), chr(0xF8FF)
pua_re = re.compile(f"[{PUA_LOW}-{PUA_HIGH}]")
paren_label_re = re.compile(
    r"\((지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\)"
)


def needs_llm_rerun(text):
    if pd.isna(text):
        return False
    return bool(pua_re.search(text)) or bool(paren_label_re.search(text))


# 체크포인트 파일을 직접 읽어서 판별한다(df_leaf_final 병합을 기다릴 필요 없음).
# 이렇게 해야 LLM 처리 셀보다 앞에 둘 수 있고, 한 번의 순차 실행(Run All)만으로
# 재실행 대상까지 자동으로 포함되어 처리된다 (CodeRabbit 리뷰 반영).
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    affected_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
    affected_idx = checkpoint_df.index[affected_mask]

    if len(affected_idx) > 0:
        checkpoint_df = checkpoint_df.drop(index=affected_idx)
        checkpoint_df.to_csv(CHECKPOINT_PATH)

    print(
        "재실행 대상(체크포인트에서 제거):",
        len(affected_idx),
        "건 -> 남은 행수:",
        len(checkpoint_df),
    )
else:
    print("체크포인트 파일 없음 - 전체 신규 실행")

재실행 대상(체크포인트에서 제거): 0 건 -> 남은 행수: 8122


In [36]:
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
CHUNK_SIZE = 200

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    # 구조 이상으로 제외된 행의 기존 체크포인트는 제거한다.
    checkpoint_df = checkpoint_df.loc[checkpoint_df.index.intersection(df_leaf_final.index)].copy()
    checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
    done_index = set(checkpoint_df.index)
    print(f"체크포인트 발견: {len(done_index)}건 이미 처리됨, 이어서 진행")
else:
    checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])
    done_index = set()

targets = df_leaf_final.index.difference(done_index)
results = {}

for i, idx in enumerate(tqdm(targets), start=1):
    row = df_leaf_final.loc[idx]
    results[idx] = clean_sentence(row["세부사업명"], row["주요내용"])

    if i % CHUNK_SIZE == 0:
        partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df.to_csv(CHECKPOINT_PATH)
        results = {}

# 남은 것 마저 저장
if results:
    partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
    checkpoint_df = pd.concat([checkpoint_df, partial])
    checkpoint_df.to_csv(CHECKPOINT_PATH)

df_leaf_final["주요내용_정제"] = checkpoint_df["주요내용_정제"]
df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~df_leaf_final["숫자보존"]).sum())
display(
    df_leaf_final[~df_leaf_final["숫자보존"]][
        ["지역", "세부사업명", "주요내용", "주요내용_정제"]
    ].head(20)
)

df_leaf_final[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세"]].head(20)

체크포인트 발견: 8122건 이미 처리됨, 이어서 진행


0it [00:00, ?it/s]

숫자 불일치(검토 대상) 건수: 0


,지역,세부사업명,주요내용,주요내용_정제


,세부사업명,주요내용,주요내용_정제,지원대상,지원내용_상세
2,저출생 인식개선 캠페인,"지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등","지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",NaN,NaN
3,어린이 안전 영상정보 인프라 구축,지원대상 : 어린이 지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,지원대상 : 어린이 지원내용 : 어린이보호구역 내 과속단속카메라 등 설치,어린이,어린이보호구역 내 과속단속카메라 등 설치
4,입양아동 가족지원,"지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정 지원내용 : 입양아동양육보조금, 심리치료비 등","지원대상 : 입양특례법에 의한 18세미만 입양아동 및 그 가정 지원내용 : 입양아동양육보조금, 심리치료비 등",입양특례법에 의한 18세미만 입양아동 및 그 가정,"입양아동양육보조금, 심리치료비 등"
5,국공립어린이집 등 보육서비스 수준제고,지원대상 : 국공립어린이집 등 지원내용 : 보육교직원 인건비 지원,지원대상 : 국공립어린이집 등 지원내용 : 보육교직원 인건비 지원,국공립어린이집 등,보육교직원 인건비 지원
6,국공립어린이집 확충,국공립어린이집 이용률 46%달성,국공립어린이집 이용률 46%달성,NaN,NaN
7,어린이집 이용 영유아 무상보육 확대,지원대상 : 어린이집 이용 영유아 지원내용 : 해당 연령 보육료,지원대상 : 어린이집 이용 영유아 지원내용 : 해당 연령 보육료,어린이집 이용 영유아,해당 연령 보육료
8,육아종합지원센터 운영,"지원대상 : 보육교직원 및 영유아 부모 지원내용 : 보육정보 및 교육, 전문상담 제공","지원대상 : 보육교직원 및 영유아 부모 지원내용 : 보육정보 및 교육, 전문상담 제공",보육교직원 및 영유아 부모,"보육정보 및 교육, 전문상담 제공"
9,가정양육수당 지원,지원대상 : 어린이집･유치원 미이용 0~86개월 미만 아동 지원내용 : 10~20만원(월) 양육수당 지원,지원대상 : 어린이집･유치원 미이용 0~86개월 미만 아동 지원내용 : 10~20만원(월) 양육수당 지원,어린이집･유치원 미이용 0~86개월 미만 아동,10~20만원(월) 양육수당 지원
10,부모 모니터링단 운영,지원대상 : 25개 자치구 지원내용 : 부모 모니터링단 운영비 지원,지원대상 : 25개 자치구 지원내용 : 부모 모니터링단 운영비 지원,25개 자치구,부모 모니터링단 운영비 지원
11,공동육아나눔터,"지원대상 : 18세 미만 자녀와 부모 지원내용 : 돌봄장소, 가족 품앗이 활동 지원","지원대상 : 18세 미만 자녀와 부모 지원내용 : 돌봄장소, 가족 품앗이 활동 지원",18세 미만 자녀와 부모,"돌봄장소, 가족 품앗이 활동 지원"


In [37]:
# 전수검토 결과를 체크포인트와 최종 데이터에 멱등 반영
REVIEW_PATH = REPORTS_DIR / f"{YEAR}_LLM_주요내용_정제_319건_전수검토.csv"
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
review_df = pd.read_csv(REVIEW_PATH, encoding="utf-8-sig")
assert len(review_df) == 319

review_counts = review_df["판정"].value_counts().to_dict()
assert review_counts == {"공백 정리": 154, "명백한 오탈자 교정": 126, "원문 유지": 39}
safe_count = review_counts["공백 정리"] + review_counts["명백한 오탈자 교정"]

key_cols = ["지역", "원본행", "세부사업명"]
keep_review = review_df.loc[review_df["판정"].eq("원문 유지"), key_cols]
assert len(keep_review) == 39
assert not keep_review.duplicated(key_cols).any()

leaf_keys = df_leaf_final.reset_index(names="_df_index")[key_cols + ["_df_index"]]
keep_matches = keep_review.merge(leaf_keys, on=key_cols, how="left", indicator=True)
assert len(keep_matches) == 39
assert keep_matches["_merge"].eq("both").all()
assert keep_matches["_df_index"].notna().all()
assert keep_matches["_df_index"].is_unique
keep_indices = keep_matches["_df_index"].tolist()

checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0, encoding="utf-8-sig")
assert checkpoint_df.index.is_unique
assert len(checkpoint_df) == 8122
assert set(checkpoint_df.index) == set(df_leaf_final.index)
assert set(keep_indices).issubset(checkpoint_df.index)

for idx in keep_indices:
    original = df_leaf_final.at[idx, "주요내용"]
    df_leaf_final.at[idx, "주요내용_정제"] = original
    checkpoint_df.at[idx, "주요내용_정제"] = original

assert df_leaf_final.loc[keep_indices, "주요내용_정제"].equals(
    df_leaf_final.loc[keep_indices, "주요내용"]
)
assert checkpoint_df.loc[keep_indices, "주요내용_정제"].equals(
    df_leaf_final.loc[keep_indices, "주요내용"]
)

checkpoint_tmp = CHECKPOINT_PATH.with_suffix(".csv.tmp")
checkpoint_df.to_csv(checkpoint_tmp, encoding="utf-8-sig")
checkpoint_check = pd.read_csv(checkpoint_tmp, index_col=0, encoding="utf-8-sig")
assert len(checkpoint_check) == 8122
assert checkpoint_check.index.is_unique
assert checkpoint_check.loc[keep_indices, "주요내용_정제"].equals(
    df_leaf_final.loc[keep_indices, "주요내용"]
)
checkpoint_tmp.replace(CHECKPOINT_PATH)

number_preservation_tests = [
    ("-10만원", "10만원", False),
    ("+10명", "10명", False),
    ("10.5%", "105%", False),
    ("1,234.5원", "1234.5원", True),
    ("3,00천원", "3,000천원", False),
    ("20-50천원", "20-50천원", True),
    ("20-50천원", "20-60천원", False),
    ("−10만원", "-10만원", True),
    ("10%", "10", False),
    (None, None, True),
    (None, "10", False),
]
for original, cleaned, expected in number_preservation_tests:
    assert numbers_preserved(original, cleaned) is expected
print(f"강화 숫자 검사 단위 테스트: {len(number_preservation_tests)}건 통과")


df_leaf_final["숫자보존"] = [
    numbers_preserved(original, cleaned)
    for original, cleaned in zip(df_leaf_final["주요내용"], df_leaf_final["주요내용_정제"])
]
number_match_count = int(df_leaf_final["숫자보존"].sum())
number_mismatch_count = int((~df_leaf_final["숫자보존"]).sum())
assert number_match_count == 8122
assert number_mismatch_count == 0

print(f"검토 CSV: {len(review_df):,}건")
print(f"안전 판정: {safe_count:,}건")
print(f"원문 유지 적용: {len(keep_indices):,}건")
print(
    f"체크포인트: {len(checkpoint_df):,}행 / 인덱스 중복 {checkpoint_df.index.duplicated().sum()}건"
)
print(f"숫자보존 일치: {number_match_count:,}건 / 불일치: {number_mismatch_count:,}건")

output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]

missing_cols = [c for c in output_cols if c not in df_leaf_final.columns]

if missing_cols:
    raise KeyError(f"df_leaf_final에 없는 칼럼: {missing_cols}")

wide_file_count = 0
wide_row_count = 0
for sido_name, group in df_leaf_final.groupby("지역"):
    out_path = get_sido_dir(sido_name) / f"{YEAR}_{sido_name}_세부사업_정제.csv"
    group[output_cols].to_csv(out_path, index=False, encoding="utf-8-sig")
    wide_file_count += 1
    wide_row_count += len(group)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")

assert wide_file_count == 17
assert wide_row_count == 8122
print(f"\nwide 저장: {wide_file_count}개 / {wide_row_count:,}행")

강화 숫자 검사 단위 테스트: 11건 통과
검토 CSV: 319건
안전 판정: 280건
원문 유지 적용: 39건
체크포인트: 8,122행 / 인덱스 중복 0건
숫자보존 일치: 8,122건 / 불일치: 0건
강원: 521행 저장 -> ../data/interim/강원/2022_강원_세부사업_정제.csv
경기: 1310행 저장 -> ../data/interim/경기/2022_경기_세부사업_정제.csv
경남: 707행 저장 -> ../data/interim/경남/2022_경남_세부사업_정제.csv
경북: 742행 저장 -> ../data/interim/경북/2022_경북_세부사업_정제.csv
광주: 204행 저장 -> ../data/interim/광주/2022_광주_세부사업_정제.csv
대구: 278행 저장 -> ../data/interim/대구/2022_대구_세부사업_정제.csv
대전: 322행 저장 -> ../data/interim/대전/2022_대전_세부사업_정제.csv
부산: 748행 저장 -> ../data/interim/부산/2022_부산_세부사업_정제.csv
서울: 227행 저장 -> ../data/interim/서울/2022_서울_세부사업_정제.csv
세종: 108행 저장 -> ../data/interim/세종/2022_세종_세부사업_정제.csv
울산: 254행 저장 -> ../data/interim/울산/2022_울산_세부사업_정제.csv
인천: 377행 저장 -> ../data/interim/인천/2022_인천_세부사업_정제.csv
전남: 552행 저장 -> ../data/interim/전남/2022_전남_세부사업_정제.csv
전북: 498행 저장 -> ../data/interim/전북/2022_전북_세부사업_정제.csv
제주: 160행 저장 -> ../data/interim/제주/2022_제주_세부사업_정제.csv
충남: 622행 저장 -> ../data/interim/충남/2022_충남_세부사업_정제.csv
충북: 492행 저장 -> ../da

In [38]:
df_leaf_final.columns.tolist()

['연도',
 '지역',
 '대분류',
 '중분류',
 '사업분류재정구분',
 '세부사업명',
 '주요내용',
 '당해예산',
 '전년도예산',
 '증감액',
 '증감율',
 '원본행',
 '주요내용_패턴',
 '주요내용_패턴_확장',
 '지원대상',
 '지원내용_상세',
 '주요내용_정제',
 '숫자보존']

In [39]:
id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]

df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

# 전년도예산 행은 실제 그 돈이 집행된 연도로 맞춰서 -1
previous_mask = df_long["예산구분"] == "전년도예산"
df_long.loc[previous_mask, "연도"] -= 1
# 증감액/증감율은 "당해 대비 전년" 개념이라 전년도 행에는 그대로 복제되면 안 됨 (CodeRabbit 지적)
df_long.loc[previous_mask, ["증감액", "증감율"]] = None

df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

assert len(df_leaf_final) == 8122
assert len(df_long) == 16244
print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf_final), "행 x 2)")
df_long.head(10)

long 변환 결과: 16244 행 (wide 8122 행 x 2)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,증감액,증감율,원본행,지원대상,지원내용_상세,주요내용_정제,예산구분,예산액
0,2022,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,농촌공동아이돌봄센터 지원,"농촌지역, 국공립+사회복지법인 어린이집 돌봄센터 운영비 지원",108.0,67.5,4587,NaN,NaN,"농촌지역, 국공립+사회복지법인 어린이집 돌봄센터 운영비 지원",당해예산,268.0
1,2021,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,농촌공동아이돌봄센터 지원,"농촌지역, 국공립+사회복지법인 어린이집 돌봄센터 운영비 지원",NaN,NaN,4587,NaN,NaN,"농촌지역, 국공립+사회복지법인 어린이집 돌봄센터 운영비 지원",전년도예산,160.0
2,2022,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,아이돌봄 서비스 지원,아이돌봄 서비스 제공,1784.0,9.8,4588,NaN,NaN,아이돌봄 서비스 제공,당해예산,20032.0
3,2021,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,아이돌봄 서비스 지원,아이돌봄 서비스 제공,NaN,NaN,4588,NaN,NaN,아이돌봄 서비스 제공,전년도예산,18248.0
4,2022,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,공동육아나눔터 운영,"공동육아 공간 제공, 장난감 도서관 운영등",112.0,11.4,4589,NaN,NaN,"공동육아 공간 제공, 장난감 도서관 운영등",당해예산,1097.0
5,2021,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,공동육아나눔터 운영,"공동육아 공간 제공, 장난감 도서관 운영등",NaN,NaN,4589,NaN,NaN,"공동육아 공간 제공, 장난감 도서관 운영등",전년도예산,985.0
6,2022,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립 어린이집 확충,국공립 어린이집 신축 1개소,-1419.0,-40.4,4590,NaN,NaN,국공립 어린이집 신축 1개소,당해예산,2097.0
7,2021,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립 어린이집 확충,국공립 어린이집 신축 1개소,NaN,NaN,4590,NaN,NaN,국공립 어린이집 신축 1개소,전년도예산,3516.0
8,2022,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립 어린이집 확충,"도내 국공립어린이집 신축, 공동주택리모델링, 장기임차, 가자재비 등",52.0,1.9,4591,NaN,NaN,"도내 국공립어린이집 신축, 공동주택리모델링, 장기임차, 가자재비 등",당해예산,2752.0
9,2021,강원,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립 어린이집 확충,"도내 국공립어린이집 신축, 공동주택리모델링, 장기임차, 가자재비 등",NaN,NaN,4591,NaN,NaN,"도내 국공립어린이집 신축, 공동주택리모델링, 장기임차, 가자재비 등",전년도예산,2700.0


In [40]:
# 시도별로 저장
for sido_name, group in df_long.groupby("지역"):
    out_path = get_sido_dir(sido_name) / f"{YEAR}_{sido_name}_세부사업_정제_long.csv"
    group.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")

강원: 1042행 저장 -> ../data/interim/강원/2022_강원_세부사업_정제_long.csv
경기: 2620행 저장 -> ../data/interim/경기/2022_경기_세부사업_정제_long.csv
경남: 1414행 저장 -> ../data/interim/경남/2022_경남_세부사업_정제_long.csv
경북: 1484행 저장 -> ../data/interim/경북/2022_경북_세부사업_정제_long.csv
광주: 408행 저장 -> ../data/interim/광주/2022_광주_세부사업_정제_long.csv
대구: 556행 저장 -> ../data/interim/대구/2022_대구_세부사업_정제_long.csv
대전: 644행 저장 -> ../data/interim/대전/2022_대전_세부사업_정제_long.csv
부산: 1496행 저장 -> ../data/interim/부산/2022_부산_세부사업_정제_long.csv
서울: 454행 저장 -> ../data/interim/서울/2022_서울_세부사업_정제_long.csv
세종: 216행 저장 -> ../data/interim/세종/2022_세종_세부사업_정제_long.csv
울산: 508행 저장 -> ../data/interim/울산/2022_울산_세부사업_정제_long.csv
인천: 754행 저장 -> ../data/interim/인천/2022_인천_세부사업_정제_long.csv
전남: 1104행 저장 -> ../data/interim/전남/2022_전남_세부사업_정제_long.csv
전북: 996행 저장 -> ../data/interim/전북/2022_전북_세부사업_정제_long.csv
제주: 320행 저장 -> ../data/interim/제주/2022_제주_세부사업_정제_long.csv
충남: 1244행 저장 -> ../data/interim/충남/2022_충남_세부사업_정제_long.csv
충북: 984행 저장 -> ../data/interim/충북/2022_충북_세부사업_정제